# Lexicon features and ablation (NRC, Empath, absolutist)
Runs after the cleaning notebook (needs `train_clean.csv`, `dev_clean.csv`, `test_clean.csv`).
Everything is scored on **dev only**. Test features are built and saved but the test set is not used for modeling.

In [ ]:
# CELL 1: Install packages
!pip install empath -q

In [ ]:
# CELL 2: Imports and load the cleaned data from the cleaning notebook
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score

# fillna instead of dropna so the split sizes stay 6006 / 1000 / 3245
train = pd.read_csv('train_clean.csv').fillna('')
dev   = pd.read_csv('dev_clean.csv').fillna('')
test  = pd.read_csv('test_clean.csv').fillna('')

print('Shapes (expect 6006 / 1000 / 3245):', train.shape[0], dev.shape[0], test.shape[0])
print('Columns:', train.columns.tolist())
print(train['labels'].value_counts())

## NRC Emotion Lexicon
Download the word-level file **NRC-Emotion-Lexicon-Wordlevel-v0.92.txt** from
https://saifmohammad.com/WebPages/NRC-Emotion-Lexicon.htm (free for research use, cite Mohammad & Turney 2013),
then upload it to Colab (the folder icon on the left) or put it in Drive. Each line is `word<TAB>emotion<TAB>0/1`.

In [ ]:
# CELL 3: Load the NRC lexicon
# Change the path if your file is somewhere else (for example in Drive)
NRC_PATH = 'NRC-Emotion-Lexicon-Wordlevel-v0.92.txt'

NRC_EMOTIONS = ['anger', 'anticipation', 'disgust', 'fear', 'joy',
                'sadness', 'surprise', 'trust', 'negative', 'positive']

nrc = {}   # word -> set of emotions
with open(NRC_PATH, encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 3 and parts[2] == '1':
            word, emotion, _ = parts
            nrc.setdefault(word, set()).add(emotion)

print('NRC words with at least one emotion:', len(nrc))
print('Example, "hopeless":', nrc.get('hopeless'))

In [ ]:
# CELL 4: NRC features = share of a post's words tied to each emotion
def nrc_features(text):
    words = str(text).split()
    n = len(words)
    counts = dict.fromkeys(NRC_EMOTIONS, 0)
    for w in words:
        for emo in nrc.get(w, ()):
            counts[emo] += 1
    return {f'nrc_{e}': (c / n if n else 0.0) for e, c in counts.items()}

def build_nrc(df):
    return pd.DataFrame([nrc_features(t) for t in df['text_clean']], index=df.index)

nrc_train, nrc_dev, nrc_test = build_nrc(train), build_nrc(dev), build_nrc(test)
print(nrc_train.head(3).round(3))

## Empath
Empath has built-in categories. I picked ones tied to the depression literature, so the feature set stays small and explainable.

In [ ]:
# CELL 5: Empath features
from empath import Empath
lexicon = Empath()

EMPATH_CATS = [
    'sadness', 'suffering', 'death', 'negative_emotion', 'positive_emotion',
    'hate', 'pain', 'violence', 'shame', 'fear', 'nervousness',
    'health', 'medical_emergency', 'sleep', 'family', 'friends', 'home', 'work'
]

def empath_features(text):
    res = lexicon.analyze(str(text), categories=EMPATH_CATS, normalize=True)
    if res is None:   # empty text
        res = {c: 0.0 for c in EMPATH_CATS}
    return {f'emp_{c}': res[c] for c in EMPATH_CATS}

def build_empath(df):
    return pd.DataFrame([empath_features(t) for t in df['text_clean']], index=df.index)

emp_train, emp_dev, emp_test = build_empath(train), build_empath(dev), build_empath(test)
print(emp_train.head(3).round(3))

In [ ]:
# CELL 6: Absolutist ratio (already computed in the cleaning notebook) + save everything
abs_train = train[['absolutist_score']]
abs_dev   = dev[['absolutist_score']]
abs_test  = test[['absolutist_score']]

# Cache so these do not need to be recomputed
for split, parts in {'train': (nrc_train, emp_train, abs_train),
                     'dev':   (nrc_dev,   emp_dev,   abs_dev),
                     'test':  (nrc_test,  emp_test,  abs_test)}.items():
    pd.concat(parts, axis=1).to_csv(f'lexicon_features_{split}.csv', index=False)
print('Saved lexicon_features_train/dev/test.csv')

## Do the features separate the classes? (train only)

In [ ]:
# CELL 7: Mean of each feature by class, on train only
all_train = pd.concat([nrc_train, emp_train, abs_train], axis=1)
by_class = all_train.groupby(train['labels']).mean().T
by_class['ratio (dep / not dep)'] = by_class['depressed'] / by_class['not depressed']
print(by_class.round(4).sort_values('ratio (dep / not dep)', ascending=False).to_string())

## Ablation on dev
Same TF-IDF (bigrams, 3000 features) and the same logistic regression every time. Only the extra features change.
The TF-IDF vectorizer is refit here on the same rows, so the baseline is comparable.

In [ ]:
# CELL 8: Build TF-IDF baseline and the scaled lexicon blocks
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=3000,
                        stop_words=None, token_pattern=r"(?u)\b\w+\b")
X_train_tfidf = tfidf.fit_transform(train['text_clean'])
X_dev_tfidf   = tfidf.transform(dev['text_clean'])

y_train = train['labels']
y_dev   = dev['labels']
y_dev_bin = (y_dev == 'depressed').astype(int)

def scaled_block(tr, dv):
    # scaler is fit on train only, so nothing from dev leaks in
    sc = StandardScaler().fit(tr)
    return csr_matrix(sc.transform(tr)), csr_matrix(sc.transform(dv)), sc

blocks = {
    'NRC':        scaled_block(nrc_train, nrc_dev),
    'Empath':     scaled_block(emp_train, emp_dev),
    'Absolutist': scaled_block(abs_train, abs_dev),
}
print('Lexicon blocks ready:', list(blocks))

In [ ]:
# CELL 9: Run the ablation (logistic regression, dev set)
def fit_and_score(name, Xtr, Xdv):
    model = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)
    model.fit(Xtr, y_train)
    preds = model.predict(Xdv)
    dep_idx = list(model.classes_).index('depressed')
    auc = roc_auc_score(y_dev_bin, model.predict_proba(Xdv)[:, dep_idx])
    return {
        'Feature set': name,
        'AUC': auc,
        'Macro F1': f1_score(y_dev, preds, average='macro'),
        'Precision (dep)': precision_score(y_dev, preds, pos_label='depressed'),
        'Recall (dep)': recall_score(y_dev, preds, pos_label='depressed'),
        '% flagged depressed': (preds == 'depressed').mean() * 100,
    }, model

def combo(names):
    tr = [X_train_tfidf] + [blocks[n][0] for n in names]
    dv = [X_dev_tfidf]   + [blocks[n][1] for n in names]
    return hstack(tr).tocsr(), hstack(dv).tocsr()

rows = []
row, lr_baseline = fit_and_score('TF-IDF only', X_train_tfidf, X_dev_tfidf)
rows.append(row)

for names in [['NRC'], ['Empath'], ['Absolutist'], ['NRC', 'Empath'], ['NRC', 'Empath', 'Absolutist']]:
    Xtr, Xdv = combo(names)
    row, _ = fit_and_score('TF-IDF + ' + ' + '.join(names), Xtr, Xdv)
    rows.append(row)

# Lexicon features alone (no words at all)
lex_tr = hstack([blocks[n][0] for n in blocks]).tocsr()
lex_dv = hstack([blocks[n][1] for n in blocks]).tocsr()
row, _ = fit_and_score('Lexicon features only (no TF-IDF)', lex_tr, lex_dv)
rows.append(row)

ablation = pd.DataFrame(rows).round(3)
print('ABLATION ON DEV SET')
print(ablation.to_string(index=False))

How to read it: rank by **AUC** first (it ignores the cutoff). Also look at `% flagged depressed`. Only 60% of dev is actually depressed, so a model flagging far more than that is just inflating recall. A feature set only helps if AUC or macro F1 goes up, not just recall.

In [ ]:
# CELL 10: Which lexicon features does the full model lean on?
Xtr_full, Xdv_full = combo(['NRC', 'Empath', 'Absolutist'])
full_model = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42).fit(Xtr_full, y_train)

lex_names = list(nrc_train.columns) + list(emp_train.columns) + list(abs_train.columns)
n_tfidf = X_train_tfidf.shape[1]
coefs = full_model.coef_[0][n_tfidf:]   # lexicon columns sit after the TF-IDF columns

# sklearn orders classes alphabetically: 'depressed' is class 0, so flip the sign
# so that positive = pushes toward depressed
if list(full_model.classes_)[0] == 'depressed':
    coefs = -coefs   # coef_ of a binary model describes classes_[1]; make "depressed" positive
coef_df = pd.Series(coefs, index=lex_names).sort_values()
print(coef_df.round(3).to_string())

coef_df.plot(kind='barh', figsize=(7, 8))
plt.title('Lexicon feature weights (positive = toward depressed)')
plt.tight_layout()
plt.savefig('lexicon_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

## Before the test set
Pick the final feature set and model from the dev table above, then **stop changing things**.
For the final run, use the commented lines below once, with the chosen blocks. The test lexicon features are already saved in `lexicon_features_test.csv`.

In [ ]:
# FINAL STEP: run ONCE, only after features and model are locked on dev
# X_test_tfidf = tfidf.transform(test['text_clean'])
# lex_blocks_test = [csr_matrix(blocks[n][2].transform(globals()[k]))
#                    for n, k in [('NRC', 'nrc_test'), ('Empath', 'emp_test'), ('Absolutist', 'abs_test')]]
# X_test = hstack([X_test_tfidf] + lex_blocks_test).tocsr()
# preds = full_model.predict(X_test)
# print(f1_score(test['labels'], preds, average='macro'), recall_score(test['labels'], preds, pos_label='depressed'))